In [57]:
import pandas as pd
import numpy as np


In [58]:
df = pd.read_csv("master_data_2020.csv")

# ==========================================
# 1. DEFINE SOURCE COLUMNS
# ==========================================
treatment = "untrustworthy_flag"
raw_vote_choice_col = "presvote20post"

# ==========================================
# 2. OVERWRITE & CLEAN TARGET OUTCOMES FROM TEXT
# ==========================================
# Ensure we capture text responses and handle any trailing whitespaces cleanly
vote_series = df[raw_vote_choice_col].astype(str).str.strip()

# 1. Fix Biden Vote (1 if explicitly voted for Biden, 0 otherwise)
df["voted_biden_2020"] = np.where(vote_series == "Joe Biden", 1, 0)

# 2. Fix Trump Vote (1 if explicitly voted for Trump, 0 otherwise)
df["voted_trump_2020"] = np.where(vote_series == "Donald Trump", 1, 0)

# 3. Create Turnout (0 if they didn't vote or skipped, 1 if they cast a ballot)
# Based on your data, non-voters show up as "Did not vote for President" or missing strings
df["turnout_2020_binary"] = np.where(
    vote_series.str.contains("did not vote|skipped|__na__", case=False, na=True),
    0,
    1
)

# Structure our clean target dictionary for the loop
outcomes = {
    "2020 Turnout": "turnout_2020_binary",
    "Voted for Biden": "voted_biden_2020",
    "Voted for Trump": "voted_trump_2020",
}

# ==========================================
# 3. COMPREHENSIVE FEATURE ENGINEERING (CONDENSED & STREAMLINED)
# ==========================================
# Process Digital Literacy Index
tf_cols = ["tf_adv", "tf_pdf", "tf_spy", "tf_wiki", "tf_cache", "tf_phishing"]
df_tf = df[tf_cols].map(
    lambda val: int(str(val).strip().split()[0])
    if pd.notna(val) and str(val).strip().split()[0].isdigit()
    else np.nan
)
df["digital_literacy_index"] = df_tf.mean(axis=1)

# Process Institutional Trust Index
conf_cols = ["conf02", "conf05", "conf07", "conf08", "conf10", "conf11", "conf12"]
trust_mapping = {"A great deal": 4, "Quite a lot": 3, "Some": 2, "Very little": 1, "None at all": 0}
df_conf = df[conf_cols].map(lambda x: trust_mapping.get(str(x).strip(), np.nan))
df["institutional_trust_index"] = df_conf.mean(axis=1)

# Linearized Political Mappings
df["ideo5_linear"] = df["ideo5"].map({"Very liberal": 1, "Liberal": 2, "Moderate": 3, "Conservative": 4, "Very conservative": 5})
df["pid7_linear"] = df["pid7"].map({"Strong Democrat": 1, "Not very strong Democrat": 2, "Lean Democrat": 3, "Independent": 4, "Lean Republican": 5, "Not very strong Republican": 6, "Strong Republican": 7})

# Linearized Social Media Scale
socmed_mapping = {
    "Less than 10 minutes per day": 1, "10–30 minutes per day": 2, "31–60 minutes per day": 3,
    "1–2 hours per day": 4, "2–3 hours per day": 5, "More than 3 hours per day": 6
}
df["socmed_use_linear"] = df["socmed_use"].astype(str).str.strip().map(socmed_mapping)

# --- NEW: Condense News Interest into a Linear Scale ---
newsint_mapping = {
    "Hardly at all": 1,
    "Only now and then": 2,
    "Some of the time": 3,
    "Most of the time": 4
}
df["newsint_linear"] = df["newsint"].astype(str).str.strip().map(newsint_mapping)

# --- NEW: Condense Internet Use Frequency into a Linear Scale ---
intuse_mapping = {
    "Less often": 1,
    "About once a day": 3,
    "Several times a day": 4,
    "A few times a week": 2  # Catching common top-tier survey labels if present
}
# Fallback to handle "Several times a day" as the top option if "Almost constantly" isn't used
df["intuse_linear"] = df["intuse"].astype(str).str.strip().map(intuse_mapping).fillna(3)

# --- NEW: Condense 2016 Vote into Trump, Clinton, and Other ---
# Modified 2016 vote processing function
def condense_2016_vote_fixed(val):
    val_str = str(val).strip()
    if val_str in ["Hillary Clinton", "Donald Trump"]:
        return val_str
    # Explicitly group non-voters and missing responses into a valid category string
    elif pd.isna(val) or val_str in ["__NA__", "nan", "Did not vote", "Did not vote for President"]:
        return "Did Not Vote"
    else:
        return "Other"

df["presvote16post_condensed"] = df["presvote16post"].apply(condense_2016_vote_fixed)

# REMOVE 'turnout16' from your categorical_covariates list!
# 'presvote16post_condensed' now handles turnout implicitly.
categorical_covariates = [
    "race4", 
    "educ4", 
    "region", 
    "votereg", 
    "presvote16post_condensed" # Will generate dummies for Clinton, Other, and Did Not Vote. 
]
# Gather all our dense numeric/linear features
linear_features = [
    "ideo5_linear", 
    "pid7_linear", 
    "socmed_use_linear", 
    "newsint_linear", 
    "intuse_linear"
]

# ==========================================
# 4. DESIGN MATRIX GENERATION (THE COVARIATES)
# ==========================================
core_cols = [treatment] + list(outcomes.values()) + ["age", "female"]
engineered_continuous = ["digital_literacy_index", "institutional_trust_index"]


# Clean complete-case alignment
all_processed_cols = core_cols + engineered_continuous + linear_features + categorical_covariates
df_clean = df[all_processed_cols].dropna().copy()

# Build the final compact X Matrix
# 1. Generate all dummies with drop_first=False so we can manually control the baselines
X_matrix = pd.get_dummies(
    df_clean[["age", "female"] + engineered_continuous + linear_features + categorical_covariates],
    columns=categorical_covariates,
    drop_first=False,
    dtype=int,
)

# 2. Define exactly one baseline reference column to drop from each categorical group
baselines_to_drop = [
    'presvote16post_condensed_Donald Trump',  # 2016 Vote baseline
    'race4_White',                            # Race baseline
    'educ4_College grad',                     # Education baseline
    'region_Midwest',                         # Region baseline
    'votereg_No'                              # Voter registration baseline (if 'No' exists, otherwise drop first manually)
]

# 3. Drop them safely from the matrix
X_matrix = X_matrix.drop(columns=baselines_to_drop, errors='ignore')

# ==========================================
# 5. FINAL AIPW INPUT EXTRACTION
# ==========================================
W = df_clean[treatment].astype(int).values
Y_turnout = df_clean[outcomes["2020 Turnout"]].astype(int).values
Y_biden = df_clean[outcomes["Voted for Biden"]].astype(int).values
Y_trump = df_clean[outcomes["Voted for Trump"]].astype(int).values

print("--- PIPELINE VERIFICATION ---")
print(f"Total valid sample size (N): {len(df_clean)}")
print(f"Total covariates in X_matrix: {X_matrix.shape[1]}")
print(f"Mean 2020 Turnout rate: {Y_turnout.mean():.2%}")
print(f"Mean unconditional Biden support: {Y_biden.mean():.2%}")
print(f"Mean unconditional Trump support: {Y_trump.mean():.2%}")

--- PIPELINE VERIFICATION ---
Total valid sample size (N): 1111
Total covariates in X_matrix: 22
Mean 2020 Turnout rate: 83.80%
Mean unconditional Biden support: 51.40%
Mean unconditional Trump support: 30.51%


In [45]:
# Quick sanity check to see if they move together
print(df_clean[["institutional_trust_index"] + linear_features].corr())

                           institutional_trust_index  ideo5_linear  \
institutional_trust_index                   1.000000      0.203918   
ideo5_linear                                0.203918      1.000000   
pid7_linear                                 0.180676      0.750795   
socmed_use_linear                           0.046778     -0.051299   
newsint_linear                              0.182814     -0.051615   
intuse_linear                              -0.009260     -0.034636   

                           pid7_linear  socmed_use_linear  newsint_linear  \
institutional_trust_index     0.180676           0.046778        0.182814   
ideo5_linear                  0.750795          -0.051299       -0.051615   
pid7_linear                   1.000000          -0.018330       -0.088150   
socmed_use_linear            -0.018330           1.000000       -0.000297   
newsint_linear               -0.088150          -0.000297        1.000000   
intuse_linear                -0.004182         